# Final summary table — TopLoc HNSW vs plain HNSW vs Exact

Reproduces the HNSW block of the paper's Table 1 for all four settings (**CAsT 2019 / CAsT 2020 × Dragon / Snowflake**).

**Inputs per setting** (edit paths in the config cell):
* `global_csv` — the `global_best_vs_best_*.csv` your per-setting analysis notebook wrote (the combined headline: best HNSW baseline vs best TopLoc, pooled across M, at that setting's tolerance).
* `exact_json` — the Exact/Flat results (`{"Flat": {...}}`).

Each setting contributes three rows per model — **Exact**, **HNSW** (best baseline), **TopLoc HNSW** (best TopLoc) — laid out as two dataset column-blocks, with speedup in the Time column.

In [2]:
import json
import numpy as np
import pandas as pd

# ------------------------------------------------------------------
# Four settings = {CAsT 2019, CAsT 2020} x {Dragon, Snowflake}.
# For each, point to:
#   global_csv : the global_best_vs_best_*.csv your analysis notebook wrote
#                (the single combined HEADLINE row at that setting's REPORT_TOL,
#                 columns: base/TopLoc M/ef/up, base/TopLoc <metric>, *_ms, speedup)
#   exact_json : the Exact/Flat results json, format {"Flat": {avg_query_time_ms,
#                MRR@10, NDCG@3, NDCG@10, ...}}
# EDIT THESE PATHS. Missing files are skipped with a warning.
# ------------------------------------------------------------------
SETTINGS = {
    ("Dragon",    "CAsT 2019"): {"global_csv": "global_best_Dragon_2019.csv",
                                  "exact_json": "exact_dragon_2019.json"},
    ("Dragon",    "CAsT 2020"): {"global_csv": "global_best_Dragon_2020.csv",
                                  "exact_json": "exact_dragon_2020.json"},
    ("Snowflake", "CAsT 2019"): {"global_csv": "global_best_Snowflake_2019.csv",
                                  "exact_json": "/home/toploc1/Datasets/toploc1/Exact_Search/Snowflake/metrics_flat_snowflake_cast2019.json"},
    ("Snowflake", "CAsT 2020"): {"global_csv": "global_best_Snowflake_2020.csv",
                                  "exact_json": "/home/toploc1/Datasets/toploc1/Exact_Search/Snowflake/metrics_flat_snowflake_cast2020.json"},
}

MODELS   = ["Dragon", "Snowflake"]
DATASETS = ["CAsT 2019", "CAsT 2020"]
METRICS  = ["MRR@10", "NDCG@3", "NDCG@10"]
SEARCHES = ["Exact", "HNSW", "TopLoc HNSW"]   # HNSW block only (add IVF later if needed)
DECIMALS = 4

## Build the table

In [5]:
def load_exact(path):
    with open(path) as f:
        d = json.load(f)
    return d.get("Flat") or next(iter(d.values()))   # tolerate any top key


def load_global(path):
    return pd.read_csv(path).iloc[0]                  # single headline row


def t_metric(v):
    return round(float(v), DECIMALS)


# cells[(model, search)][(dataset, column)] = value
cells = {}
detail = []   # which M / config each winner used

for (model, dataset), paths in SETTINGS.items():
    # ---- Exact / Flat ----
    try:
        ex = load_exact(paths["exact_json"])
        d = cells.setdefault((model, "Exact"), {})
        for mt in METRICS:
            d[(dataset, mt)] = t_metric(ex[mt])
        #d[(dataset, "Time")] = f"{float(ex['avg_query_time_ms']):.1f}"
    except FileNotFoundError:
        print(f"[skip] exact json not found: {paths['exact_json']}")

    # ---- HNSW baseline + TopLoc HNSW (from the global best CSV) ----
    try:
        g = load_global(paths["global_csv"])
        dh = cells.setdefault((model, "HNSW"), {})
        dt = cells.setdefault((model, "TopLoc HNSW"), {})
        for mt in METRICS:
            dh[(dataset, mt)] = t_metric(g[f"base {mt}"])
            dt[(dataset, mt)] = t_metric(g[f"TopLoc {mt}"])
        dh[(dataset, "Time")] = f"{float(g['base ms']):.3f}   (-)"
        dt[(dataset, "Time")] = (f"{float(g['TopLoc ms']):.3f}   "
                                 f"({float(g['speedup']):.1f}x)")
        detail.append({"Model": model, "Dataset": dataset,
                       "HNSW": f"M={int(g['base M'])}, ef={int(g['base ef'])}",
                       "TopLoc HNSW": f"M={int(g['TopLoc M'])}, up={int(g['TopLoc up'])}, "
                                      f"ef={int(g['TopLoc ef'])}",
                       "TOL": g.get("TOL", np.nan),
                       "speedup": round(float(g["speedup"]), 2)})
    except FileNotFoundError:
        print(f"[skip] global csv not found: {paths['global_csv']}")

# ---- assemble the paper-style table ----
col_index = pd.MultiIndex.from_product([DATASETS, METRICS + ["Time"]])
row_tuples = [(m, s) for m in MODELS for s in SEARCHES if (m, s) in cells]
row_index = pd.MultiIndex.from_tuples(row_tuples, names=["Model", "Search"])

tbl = pd.DataFrame(index=row_index, columns=col_index, dtype=object)
for (model, search), cc in cells.items():
    for col, val in cc.items():
        tbl.loc[(model, search), col] = val
tbl = tbl.where(pd.notna(tbl), "")     # blank instead of NaN for missing settings

print("Final comparison - TopLoc HNSW vs plain HNSW vs Exact")
print("Time column: (-) = reference; (Nx) = speedup vs the HNSW row above it.\n")
display(tbl)
tbl.to_csv("final_summary_table.csv")

if detail:
    print("\nWinning configurations (which M / ef / up each global winner used):")
    display(pd.DataFrame(detail).set_index(["Model", "Dataset"]))

[skip] exact json not found: exact_dragon_2019.json
[skip] global csv not found: global_best_Dragon_2019.csv
[skip] exact json not found: exact_dragon_2020.json
[skip] global csv not found: global_best_Dragon_2020.csv
Final comparison - TopLoc HNSW vs plain HNSW vs Exact
Time column: (-) = reference; (Nx) = speedup vs the HNSW row above it.



CAsT 2019                                 CAsT 2020  \
                         MRR@10  NDCG@3 NDCG@10            Time    MRR@10   
Model     Search                                                            
Snowflake Exact          0.8158  0.5501   0.502                    0.7885   
          HNSW           0.8129  0.5493  0.5017     3.030   (-)    0.7838   
          TopLoc HNSW    0.8129  0.5493  0.5019  0.762   (4.0x)    0.7855   

                                                       
                       NDCG@3 NDCG@10            Time  
Model     Search                                       
Snowflake Exact        0.5065  0.4741                  
          HNSW         0.5048  0.4732     0.395   (-)  
          TopLoc HNSW  0.5099  0.4717  0.095   (4.2x)


Winning configurations (which M / ef / up each global winner used):


HNSW         TopLoc HNSW    TOL  speedup
Model     Dataset                                                    
Snowflake CAsT 2019  M=32, ef=512  M=32, up=2, ef=256  0.000     3.98
          CAsT 2020  M=32, ef=128   M=64, up=2, ef=16  0.005     4.16

## Styled render (optional)

In [4]:
# Optional: styled HTML render (bold TopLoc rows), saved to disk.
try:
    sty = (tbl.style
             .set_caption("TopLoc HNSW vs HNSW vs Exact - final comparison")
             .set_table_styles([{"selector": "th", "props": [("text-align", "center")]}]))
    html = sty.to_html()
    with open("final_summary_table.html", "w") as f:
        f.write(html)
    print("wrote final_summary_table.html")
    display(sty)
except Exception as e:
    print("styler unavailable:", e)

wrote final_summary_table.html


## Notes

The HNSW and TopLoc-HNSW numbers come straight from each setting's global-best CSV, so this notebook does no re-analysis — it only assembles. The `Time` cell shows `(-)` for the HNSW reference and `(Nx)` for TopLoc's speedup against it; Exact shows its raw ms. The winning-config table records which M each winner came from (baseline and TopLoc may differ). To add the IVF block later, extend `SEARCHES` and feed the IVF equivalents the same way.